# 03 · FAISS 检索对比：Flat / IVF / HNSW 三种索引实战

本 notebook 用**同一批模拟向量**，把 FAISS 里三种最常用的向量索引各跑一遍，重点对比三个指标：

1. **构建耗时** —— 把数据"装进"索引要花多久
2. **搜索耗时** —— 拿到一个查询后，找到相似向量要花多久
3. **召回率** —— 近似算法找到的结果里，有多少是"真正正确"的

> 运行前先安装依赖：`pip install faiss-cpu numpy`

> **一句话总结：Flat 最准但慢，IVF 快但可能漏，HNSW 又快又准但构建慢。**


## 一、三种方法的思路（用"图书馆找书"建立直觉）

向量检索的本质：给一个「查询向量」，从海量「底库向量」里找出最相似的 K 个。

| 方法 | 核心思路 | 图书馆类比 | 优点 | 缺点 |
|------|---------|-----------|------|------|
| **Flat** 暴力搜索 | 和底库**每一个**向量都算一遍距离，取最近 K 个 | 一本一本翻完所有书 | 100% 精确 | 数据一多就慢 |
| **IVF** 倒排文件 | 先把向量聚成 N 个"桶"，搜索时只钻最近的几个桶 | 先把书按类分柜，只翻相关柜子 | 快很多、省内存 | 可能漏、要先训练 |
| **HNSW** 分层图 | 构建多层"小世界"网络图，逐层跳转逼近目标 | 先定位"城市→区→街道"逐级导航 | 又快又准 | 构建慢、吃内存 |

**关键结论**：Flat 是"标准答案"，IVF / HNSW 是"近似答案"。所以后面用**召回率**（近似结果里有多少个真的进了精确结果）来衡量近似算法牺牲了多少精度。

下面三小节逐个实现这三种方法，最后在主程序里统一跑一遍并对比。


In [3]:
import faiss
import numpy as np
import time


# ══════════════════════════════════════════════════════════
# ⭐ 方法一：Flat（暴力搜索）—— 最"笨"但最准的基准线
# ══════════════════════════════════════════════════════════
def flat(d, xb, xq, k):
    """
    暴力搜索：把查询向量和底库里"每一个"向量都算一遍距离，挑出最近的 k 个。

    —— 直觉类比：在图书馆里一本一本翻完所有书，绝不出错，但数据一多就慢。

    参数：
        d  : 向量维度（每个向量由多少个数字组成，本例 = 128）
        xb : 底库向量（被搜索的"海量数据"，本例 10 万个）
        xq : 查询向量（你想找相似的那个"问题"，本例 5 个）
        k  : 要找最近的几个（本例 = 6）
    返回：最相似的 k 个向量的"编号"，作为后续算召回率时的"标准答案"
    """
    print("--- 1. 精确近邻 (NN) 搜索 ---")
    t0 = time.monotonic()  # 记录开始时刻。用 monotonic（单调时钟）而非 time()，可避免系统改时间导致计时不准

    # 创建 Flat 索引：d 表示向量维度；L2 表示用"欧氏距离"（两点间直线距离）来衡量相似度
    index = faiss.IndexFlatL2(d)

    # 把 10 万个底库向量塞进索引。Flat 无需训练，add 就是全部构建工作，所以几乎瞬间完成
    index.add(xb)
    t1 = time.monotonic()  # 构建完成时刻

    # 搜索：拿 5 个查询向量，各自找出最近的 k 个
    #   D = 距离(Distance)，越小越相似
    #   I = 索引编号(Id)，即"最相似向量在底库中的位置"
    D, I = index.search(xq, k)  # D(距离); I (索引)
    t2 = time.monotonic()  # 搜索完成时刻

    print(f"构建耗时: {(t1 - t0) * 1000:.4f} ms")  # 乘以 1000 把"秒"换算成"毫秒"; :.4f 保留 4 位小数
    print(f"搜索耗时: {(t2 - t1) * 1000:.4f} ms")
    # 想看具体结果就取消下面两行注释：
    # print(f"结果 (索引):\n{I}\n")
    # print(f"结果 (距离):\n{D}\n")
    return I   # 返回精确结果，作为 IVF / HNSW 对比的"标准答案"


In [7]:
# ══════════════════════════════════════════════════════════
# ⭐ 方法二：IVF（倒排文件）—— 先分桶，再只搜最近的几个桶
# ══════════════════════════════════════════════════════════
def ivf(d, xb, xq, k):
    """
    倒排文件：先把所有向量聚成 nlist 个"桶"(cluster)，搜索时只钻进离查询最近的
    几个桶里找，不用遍历全部数据，所以快很多。

    —— 直觉类比：先把书按类别分到不同柜子，找书时只翻最相关的几个柜子。
    代价：可能因为没翻到正确的柜子而漏掉答案，所以属于"近似"搜索。
    """
    print("--- 2. IVF ---")
    t0 = time.monotonic()

    # 步骤 1: 创建 IVF 索引
    nlist = 20                             # 分成 20 个桶（聚类中心的数量）
    quantizer = faiss.IndexFlatL2(d)       # "量尺"：用来给向量做聚类/找桶的底层索引
    index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_L2)

    # 步骤 2: 训练（IVF 特有！）—— 就是算出这 20 个桶各自的"中心点"在哪
    index_ivf.train(xb)

    # 步骤 3: 添加向量 —— 把每个向量分派到离它最近的那个桶里
    index_ivf.add(xb)
    t1 = time.monotonic()  # 训练 + 添加都算"构建"时间

    # 步骤 4: 设置 nprobe（探查桶的数量）：搜索时查看离查询最近的 15 个桶
    #    nprobe 越大 → 翻的桶越多 → 越准但越慢；越小 → 越快但越容易漏
    index_ivf.nprobe = 15

    # 步骤 5: 搜索
    D, I = index_ivf.search(xq, k)
    t2 = time.monotonic()

    print(f"训练+构建耗时: {(t1 - t0) * 1000:.4f} ms")
    print(f"搜索耗时 (nprobe={index_ivf.nprobe}): {(t2 - t1) * 1000:.4f} ms")
    # print(f"结果 (索引):\n{I}\n")
    return I


In [5]:
# ══════════════════════════════════════════════════════════
# ⭐ 方法三：HNSW（分层可导航小世界图）—— 像"城市导航"一样逐级逼近目标
# ══════════════════════════════════════════════════════════
def hnsw(d, xb, xq, k):
    """
    分层图：把向量组织成一张多层的"小世界"网络图，上层稀疏、下层密集。
    搜索时从最上层大步跳跃开始，一层层往下、越走越细，最终逼近目标。

    —— 直觉类比：先看"哪个城市"→ 再看"哪个区"→ 最后看"哪条街"。
    优点：又快又准；缺点：构建图很慢、也比较吃内存。
    """
    print("--- 3. HNSW ---")
    t0 = time.monotonic()

    # 步骤 1: 构建 HNSW 索引
    #   M = 32：每个节点连接多少个邻居。M 越大 → 图越密 → 越准，但越占内存、构建越慢
    index_hnsw = faiss.IndexHNSWFlat(d, 32)

    #   efConstruction = 96：构建索引时维护的"候选列表"长度，越大图质量越好，但构建越慢
    index_hnsw.hnsw.efConstruction = 96

    # 步骤 2: 添加数据 —— HNSW 最耗时的部分，因为要一边插入一边动态建图
    index_hnsw.add(xb)
    t1 = time.monotonic()

    # efSearch = 256：搜索时维护的"候选列表"长度，越大越准但越慢
    index_hnsw.hnsw.efSearch = 256

    # 步骤 3: 搜索
    D_hnsw, I_hnsw = index_hnsw.search(xq, k)
    t2 = time.monotonic()

    print(f"构建耗时: {(t1 - t0) * 1000:.4f} ms")
    print(f"搜索耗时 (efSearch={index_hnsw.hnsw.efSearch}): {(t2 - t1) * 1000:.4f} ms")
    # print(f"结果 (索引):\n{I_hnsw}\n")
    return I_hnsw


In [11]:
# ══════════════════════════════════════════════════════════
# ⭐ 主程序：造数据 → 跑三种方法 → 算召回率对比
# ══════════════════════════════════════════════════════════
if __name__ == '__main__':

    # ---- 第一步：造模拟数据 ----
    d = 128                       # 每个向量 128 维
    nb = 100000                   # 底库里放 10 万个向量
    np.random.seed(12)            # 固定随机种子，让每次运行生成一模一样的数据，方便复现对比
    xb = np.random.random((nb, d)).astype('float32')  # 底库：(100000, 128) 的随机数；转 float32 是因为 faiss 只接受 float32
    # print(xb)

    # ---- 第二步：造查询向量 ----
    nq = 5                        # 5 个查询
    xq = np.random.random((nq, d)).astype('float32')  # 查询：(5, 128)
    k = 6 # 每个查询找最近的 6 个邻居

    # ---- 第三步：依次跑三种方法 ----
    flat_results = flat(d, xb, xq, k)  # ① 精确暴力搜索（作为"标准答案"）
    ivf_results = ivf(d, xb, xq, k)  # ② 倒排文件（近似搜索）
    hnsw_results = hnsw(d, xb, xq, k)  # ③ 分层小世界图（近似搜索）

    # ---- 第四步：召回率 —— 衡量"近似算法"到底有多接近"精确答案" ----
    # 召回率 = 近似结果里，有几个真正出现在精确结果里 / 要找的个数 k
    def recall(flat_results, ann_results):
        assert flat_results.shape == ann_results.shape  # 两者形状必须一致，否则报错
        # 逐个查询统计"近似结果 ∩ 精确结果"有多少个共同元素（axis=1 表示按每一行=每个查询来求和）
        common_elements = (np.isin(ann_results, flat_results).sum(axis=1))
        return np.mean(common_elements) / k  # 取所有查询的平均，再除以 k 得到百分比

    print("--- 性能对比 ---")
    print(f"IVF 召回率 (与精确结果相比): {recall(flat_results, ivf_results):.2%}")
    print(f"HNSW 召回率 (与精确结果相比): {recall(flat_results, hnsw_results):.2%}")


--- 1. 精确近邻 (NN) 搜索 ---
构建耗时: 0.0000 ms
搜索耗时: 16.0000 ms
--- 2. IVF ---
训练+构建耗时: 593.0000 ms
搜索耗时 (nprobe=15): 16.0000 ms
--- 3. HNSW ---
构建耗时: 10016.0000 ms
搜索耗时 (efSearch=256): 0.0000 ms
--- 性能对比 ---
IVF 召回率 (与精确结果相比): 50.00%
HNSW 召回率 (与精确结果相比): 73.33%


## 三、总结：三种索引怎么选？

| 场景 | 推荐 | 理由 |
|------|------|------|
| 数据量小（几万以内）、要 100% 精确 | **Flat** | 暴力搜索够快，没必要上近似 |
| 数据量大、内存有限、能接受轻微损失 | **IVF** | 省内存，但需要先 train |
| 数据量大、要"快 + 准"、内存充足 | **HNSW** | 无需训练、检索快，代价是构建慢 |

**关键参数速查（越大越准、越慢）：**
- `IVF`：`nlist`（分几个桶）、`nprobe`（搜索时探查几个桶）
- `HNSW`：`M`（每个节点连几个邻居）、`efConstruction`（构建质量）、`efSearch`（搜索精度）
